# 鋳造の多期チャージ配合計画

電気炉(EAF)を持つ鋳物工場の「溶解計画係」が、数日〜1週間の計画期間について、各時間帯(期)に
「どの注文グレードの溶湯を、何回(整数)、1回あたり何トン(連続)溶かし、その各チャージに
スクラップ在庫のどのロットを何トン投入するか」を決める。既存の易しい版
(`foundry_charge_mix.py`、単一チャージ・2原料)を、多期・複数注文・在庫品質ばらつき・
電力時間帯料金まで精緻化したもの。

この問題には**2種類の双線形**が同時に効いている:

1. **ヒート回数(整数)× ヒートサイズ(連続) = 溶解量**。電気炉は1回の溶解(ヒート)ごとに
   段取り(通電立上げ・出湯)が要る離散操作なので回数は整数、1回の装入量は炉容量の範囲で連続。
2. **溶湯組成(濃度)× 溶解量/配分量**。よく撹拌された1つの溶湯の炭素・銅濃度は装入配合の
   加重平均で決まり(濃度×質量)、1つの溶湯を複数注文へ配分するので同じ期に打つ注文が
   共通の溶湯組成を通じて結合する(プーリング型の強い非凸)。

各制約の業務的意味:

- **スクラップ在庫の品質ばらつき**: 回収スクラップはロットごとに炭素・銅の含有率が異なり
  在庫量も限られる。銅は精錬で除去できないため、高銅スクラップの使用がグレード上限で縛られる
  (トランプエレメント問題)。在庫は全期で共有(時間結合)。
- **成分規格・納期**: 配分先の注文は炭素規格窓・銅上限・納期を持つ。
- **電力の時間帯料金**: 夜間安・昼間高の時間帯料金があり、安い時間帯にまとめて溶かす誘因が
  生まれる。
- **外注バックストップ**: 自社溶解で納期を満たせない分は規格適合品の外注購入で埋める
  (常に実行可能性を担保)。

題材: `samples/manufacturing_and_blending/foundry_charge_mix_multiperiod.py`。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/manufacturing_and_blending"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import foundry_charge_mix_multiperiod as fc

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

既定scaleはロット12×注文8×期8。`melt[t] == n[t]*s[t]`(整数×連続)と
`carbon_bal_t`/`copper_bal_t`(濃度×質量)が非線形の中心。

In [2]:
m0 = fc.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}(うち整数 {m0.getNIntVars()})、制約 {m0.getNConss()}"
      f"(うち非線形 {nl})")
kinds = {}
for c in m0.getConss():
    if c.isNonlinear():
        k = c.name.rsplit("_", 1)[0]
        kinds[k] = kinds.get(k, 0) + 1
print("非線形制約の内訳:", kinds)
del m0

変数 264(うち整数 8)、制約 252(うち非線形 216)
非線形制約の内訳: {'melt_def': 8, 'carbon_bal': 8, 'copper_bal': 8, 'carb_lo_0': 8, 'carb_hi_0': 8, 'cu_hi_0': 8, 'carb_lo_1': 8, 'carb_hi_1': 8, 'cu_hi_1': 8, 'carb_lo_2': 8, 'carb_hi_2': 8, 'cu_hi_2': 8, 'carb_lo_3': 8, 'carb_hi_3': 8, 'cu_hi_3': 8, 'carb_lo_4': 8, 'carb_hi_4': 8, 'cu_hi_4': 8, 'carb_lo_5': 8, 'carb_hi_5': 8, 'cu_hi_5': 8, 'carb_lo_6': 8, 'carb_hi_6': 8, 'cu_hi_6': 8, 'carb_lo_7': 8, 'carb_hi_7': 8, 'cu_hi_7': 8}


`melt_def`(整数×連続)は期の数だけ、`carbon_bal`/`copper_bal`(濃度×質量)と
`carb_lo`/`carb_hi`/`cu_hi`(配分量×濃度の成分規格)は注文×期の組み合わせだけ現れる。
後者の方が圧倒的に本数が多く、注文間で溶湯組成 `cc[t]`/`cu[t]` を共有する**プーリング型の
結合**を作っている。

## 2. 診断

In [3]:
report = mk.analyze(lambda: fc.build_model("default"), name="foundry-charge-mix",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 空間分枝の双対寄与:", f"{report.metrics.get('spatial_share', 0):.1%}",
      "/ 停滞区間数:", report.metrics.get("n_stalls"))

[foundry-charge-mix] 検出症状 1件:
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化

gap: 52.6% / 空間分枝の双対寄与: 89.5% / 停滞区間数: 3


30秒でgapが50%超残り、`dual_stall`(双対境界の改善停滞)が発火する。空間分枝(連続変数への
分枝)が双対境界改善の9割近くを占め、離散側(ヒート回数)の分枝だけでは gap は縮まらない —
プーリング型の結合(溶湯組成の共有)が支配的なボトルネックであることを示唆する。中身を見る。

## 3. 診断の中身を見る — 帰属分析と違反の内訳

`mk.collectors.attribution`(双対境界の改善をどの分枝変数に帰属させたか)と
`mk.collectors.violation`(ルートLP緩和解での非線形制約違反)を直接叩く。

In [4]:
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind, gain_by_variable
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

m1 = fc.build_model("default")
d1, summ1 = solve_and_attribute(m1, time_limit=30, gap_limit=0.01)
gk = gain_by_kind(d1)
gk

,kind,dual_gain
2,spatial,1216.379546
0,integer,147.505167
1,root,0.000000


In [5]:
fig = go.Figure(layout=base_layout(
    "双対境界の改善の帰属(分枝の種類別、合計に対する割合)",
    "分枝の種類", "双対境界の押し上げ量(合計)", height=380))
fig.add_trace(go.Bar(x=gk["kind"], y=gk["dual_gain"],
    marker_color=[C["warn"] if k == "spatial" else C["s1"] for k in gk["kind"]],
    text=[f"{v:.1f}" for v in gk["dual_gain"]], textposition="outside"))
show(fig)

In [6]:
m2 = fc.build_model("default")
vdf = collect_root_violations(m2)
vt = violation_by_type(vdf)
vt

,ctype,mean_rel,max_rel,n
17,carbon_bal,2.479459e-01,9.959449e-01,8
30,melt_def,2.479085e-01,9.917355e-01,8
14,carb_lo_5,1.360872e-01,8.430733e-01,8
13,carb_lo_4,1.227280e-01,9.818237e-01,8
16,carb_lo_7,1.197716e-01,9.581730e-01,8
11,carb_lo_2,1.192099e-01,9.536788e-01,8
12,carb_lo_3,1.149840e-01,9.198718e-01,8
24,cu_hi_4,1.149612e-01,9.196899e-01,8
25,cu_hi_5,1.148606e-01,6.956723e-01,8
15,carb_lo_6,1.120972e-01,8.967774e-01,8


In [7]:
fig = go.Figure(layout=base_layout(
    "非線形制約タイプ別の相対違反(ルートLP緩和解、平均降順)",
    "制約タイプ", "相対違反(平均)", height=380))
fig.add_trace(go.Bar(x=vt["ctype"], y=vt["mean_rel"],
    marker_color=C["warn"], text=[f"{v:.2f}" for v in vt["mean_rel"]],
    textposition="outside"))
fig.update_xaxes(tickangle=-30)
show(fig)

`spatial`(連続変数への空間分枝)が双対境界改善のほとんどを稼いでおり、`discrete`
(ヒート回数などの整数分枝)の寄与はごくわずか。違反の内訳では `carbon_bal`(濃度×質量)が
最も大きい — ヒート回数×サイズの整数×連続積(`melt_def`)よりも、溶湯組成の共有(プーリング型)
の方が支配的なボトルネックであることが裏付けられた。とはいえ `melt_def` は
`mk.linearize_product` がそのまま当てはまる形(整数×連続)なので、次にこれを厳密線形化して
どこまで効くか実測する。

## 4. 改善を試す — `melt = n・s` を `mk.linearize_product` で厳密線形化

`melt[t] == n[t]*s[t]` はヒート回数(整数、0..4)× ヒートサイズ(連続、0..30トン)の積で、
[整数×連続の厳密線形化](../improve/01_linearize_product.ipynb) と同じ整数×連続パターン。
`mk.linearize_product` で厳密な線形表現を作り、既存の `melt[t]` へ等号で結んで緩和を締める
(元の双線形制約はそのまま残すため、緩和領域は両方の交わり = 厳密線形表現の側に締まる)。

In [8]:
def build_linearized(scale="default"):
    '''melt[t] = n[t]*s[t] を mk.linearize_product で厳密線形化した変種。'''
    m = fc.build_model(scale)
    n, s, melt = m.data["n"], m.data["s"], m.data["melt"]
    nI, nO, nT = m.data["dims"]
    for t in range(nT):
        ns = mk.linearize_product(m, n[t], s[t], 0, fc.N_HEAT_MAX, 0.0, fc.HEAT_MAX,
                                  f"ns_{t}")
        m.addCons(melt[t] == ns, name=f"melt_tight_{t}")
    return m

mb = fc.build_model("small"); mb.hideOutput(); mb.optimize()
ml = build_linearized("small"); ml.hideOutput(); ml.optimize()
print(f"baseline   : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"linearized : obj={ml.getObjVal():.2f}  status={ml.getStatus()}")

baseline   : obj=36556.95  status=optimal
linearized : obj=36556.95  status=optimal


In [9]:
df = mk.compare_variants(
    {"baseline(bilinear melt)": lambda: fc.build_model("default"),
     "linearized(melt)":        lambda: build_linearized("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline(bilinear melt),127171.965167,128499.969452,0.527073,3936
1,linearized(melt),133017.660351,134062.821207,0.489092,3041


In [10]:
base = df.loc[df["variant"] == "baseline(bilinear melt)"].iloc[0]
lin  = df.loc[df["variant"] == "linearized(melt)"].iloc[0]
labels = ["baseline", "linearized(melt)"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], lin["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, lin["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], lin["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="melt=n・s の厳密線形化 before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {lin['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {lin['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(lin['nodes'])}")

root_dual : 127172.0 -> 133017.7
final_gap : 52.7% -> 48.9%
nodes     : 3936 -> 3041


**正直な検証結果**: 3節の帰属分析が示した通り、この問題のgapの主因は `melt=n・s`
(整数×連続、期の数だけ)ではなく、注文間で共有される溶湯組成 `carbon_bal`/`carb_lo`/`carb_hi`
(濃度×質量、注文×期の数だけ)というプーリング型の結合である。`melt` の厳密線形化は該当箇所の
緩和を締めるが、支配的なボトルネックには触れないため、既定scaleでの `root_dual`/`final_gap`/
`nodes` の改善は限定的になりうる(実測値は上のグラフの通り)。この結果自体が、
「3節で真因を特定してから4節の手を打つ」という診断→改善のワークフローの価値を裏付けている
— melt側だけを見て `mk.linearize_product` を適用しても、真因(プーリング型の溶湯組成結合)には
届かないことがある。

## まとめ

- この問題は**2種類の双線形が共存**する: (1) ヒート回数×サイズ(整数×連続、局所的)、
  (2) 溶湯組成×質量/配分量(濃度×流量、注文間で共有されプーリング型に結合)。
  帰属分析・違反分析はどちらが支配的かを実測で切り分ける手段になる —
  この題材では(2)が主因だった。
- 実務的には、これは「限られた高品質スクラップ(低銅)をどの注文にどう配分し、いつ電力の
  安い時間帯にまとめて溶かすか」という溶解計画そのものであり、注文間の溶湯組成の共有は
  近似ではなく「1つの炉から複数の鋳物を作る」運用実態に由来する。
- 教訓: `weak_relaxation`/`dual_stall` が発火しても、「どの非線形項が真因か」を
  attribution/violation で特定せずに手近な `mk.linearize_product` を当てるだけでは
  効果が限られることがある。

関連: [整数×連続の厳密線形化](../improve/01_linearize_product.ipynb) /
[手法ガイド 1](../../playbook/01-linearize.md) /
API [`mk.linearize_product`](../../api/transforms.md)